In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from functools import reduce

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

SOURCE_TABLES = {
    "standard": f"{SILVER_SCHEMA}.team_standard_stats",
    "shooting": f"{SILVER_SCHEMA}.team_shooting_stats",
    "goalkeeping": f"{SILVER_SCHEMA}.team_goalkeeping_stats",
    "playing_time": f"{SILVER_SCHEMA}.team_playing_time_stats",
    "miscellaneous": f"{SILVER_SCHEMA}.team_miscellaneous_stats",
    "standings": f"{SILVER_SCHEMA}.team_standings_stats",
}

DIMENSION_TABLES = {
    "team": f"{GOLD_SCHEMA}.dim_team",
    "competition": f"{GOLD_SCHEMA}.dim_competition",
    "season": f"{GOLD_SCHEMA}.dim_season",
}

GOLD_TABLE = f"{GOLD_SCHEMA}.fact_team_season"


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

SILVER_KEYS = [
    "team_key",
    "league_code",
    "season_start_year",
]

BUSINESS_KEY = [
    "team_key",
    "competition_key",
    "season_key",
]

REQUIRED_GOLD_COLUMNS = BUSINESS_KEY + [
    "matches_played",
    "matches_won",
    "matches_drawn",
    "matches_lost",
    "points",
    "goals_scored",
    "goals_conceded",
    "goal_difference",
]

# Map each source column to its business-oriented Gold name.
# Fields that describe the same metric in multiple sources
# are not loaded multiple times.

METRIC_MAPPING = {
    "standings": {
        "league_position": "league_position",
        "matches_played": "matches_played",
        "matches_won": "matches_won",
        "matches_drawn": "matches_drawn",
        "matches_lost": "matches_lost",
        "points": "points",
        "points_per_match": "points_per_match",
        "goals_scored": "goals_scored",
        "goals_conceded": "goals_conceded",
        "goal_difference": "goal_difference",
        "average_attendance": "average_attendance",
    },
    "standard": {
        "assists": "assists",
        "non_penalty_goals": "non_penalty_goals",
        "penalty_goals": "penalty_goals",
        "penalty_kicks_attempted": "penalty_kicks_attempted",
        "possession_percentage": "possession_percentage",
        "goals_per_90": "goals_per_90",
        "assists_per_90": "assists_per_90",
        "non_penalty_goals_per_90": "non_penalty_goals_per_90",
    },
    "shooting": {
        "shots": "shots",
        "shots_on_target": "shots_on_target",
        "shots_per_90": "shots_per_90",
        "shots_on_target_per_90": "shots_on_target_per_90",
        "shots_on_target_percentage": "shots_on_target_percentage",
        "goals_per_shot": "goals_per_shot",
        "goals_per_shot_on_target": "goals_per_shot_on_target",
    },
    "goalkeeping": {
        "goalkeepers_used_count": "goalkeepers_used_count",
        "saves": "saves",
        "save_percentage": "save_percentage",
        "clean_sheets": "clean_sheets",
        "clean_sheet_percentage": "clean_sheet_percentage",
        "shots_on_target_against": "shots_on_target_against",
        "penalty_kicks_faced": "penalty_kicks_faced",
        "penalty_kicks_saved": "penalty_kicks_saved",
        "penalty_kicks_allowed": "penalty_kicks_allowed",
        "penalty_kick_save_percentage": "penalty_kick_save_percentage",
    },
    "playing_time": {
        "players_used_count": "players_used_count",
        "average_age": "average_age",
        "minutes_per_match": "minutes_per_match",
        "substitute_appearances": "substitute_appearances",
        "unused_substitute_appearances": "unused_substitute_appearances",
    },
    "miscellaneous": {
        "yellow_cards": "yellow_cards",
        "red_cards": "red_cards",
        "second_yellow_cards": "second_yellow_cards",
        "fouls_committed": "fouls_committed",
        "fouls_drawn": "fouls_drawn",
        "tackles_won": "tackles_won",
        "interceptions": "interceptions",
        "crosses": "crosses",
        "offsides": "offsides",
        "own_goals": "own_goals",
        "penalty_kicks_won": "penalty_kicks_won",
        "penalty_kicks_conceded": "penalty_kicks_conceded",
    },
}


# ---------------------------------------------------------------------------
# Validate source tables and schemas
# ---------------------------------------------------------------------------

silver_dfs = {}

for source_name, table_name in SOURCE_TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

    df = spark.table(table_name)

    required_columns = (
        SILVER_KEYS
        + list(METRIC_MAPPING[source_name].keys())
    )

    validate_columns_exist(
        df,
        required_columns,
    )

    metrics = validate_required_not_null(
        df,
        SILVER_KEYS,
        fail_on_null=True,
    )

    if metrics["row_count"] == 0:
        raise ValueError(
            f"Required Silver table is empty: {table_name}"
        )

    # Each source must contain one record per team-season-league.

    duplicates = (
        df.groupBy(*SILVER_KEYS)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"Duplicate business keys found in {table_name}"
        )

    silver_dfs[source_name] = df

    print(
        f"Source validated: {source_name}, "
        f"rows={metrics['row_count']}"
    )


# ---------------------------------------------------------------------------
# Validate consistency of FBref business keys
# ---------------------------------------------------------------------------

# All six FBref datasets should contain exactly the same
# team, competition and season combinations.
#
# This was confirmed during profiling, but is validated again
# because future Silver refreshes may introduce differences.

base_keys_df = (
    silver_dfs["standings"]
    .select(*SILVER_KEYS)
    .distinct()
)

base_count = base_keys_df.count()

for source_name, df in silver_dfs.items():

    current_keys_df = (
        df.select(*SILVER_KEYS)
        .distinct()
    )

    missing_keys = (
        base_keys_df
        .join(
            current_keys_df,
            on=SILVER_KEYS,
            how="left_anti",
        )
        .count()
    )

    extra_keys = (
        current_keys_df
        .join(
            base_keys_df,
            on=SILVER_KEYS,
            how="left_anti",
        )
        .count()
    )

    if missing_keys > 0 or extra_keys > 0:
        raise ValueError(
            f"FBref key mismatch for {source_name}: "
            f"missing={missing_keys}, extra={extra_keys}"
        )

print(
    "FBref business keys validated across all sources: "
    f"{base_count} unique team-season combinations."
)


# ---------------------------------------------------------------------------
# Validate existing Gold dimensions
# ---------------------------------------------------------------------------

for dimension_name, table_name in DIMENSION_TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Gold dimension does not exist: "
            f"{table_name}"
        )

dimension_keys = {
    "team": "team_key",
    "competition": "competition_key",
    "season": "season_key",
}

dimension_dfs = {}

for dimension_name, key_column in dimension_keys.items():

    df = spark.table(
        DIMENSION_TABLES[dimension_name]
    )

    validate_columns_exist(
        df,
        [key_column],
    )

    validate_required_not_null(
        df,
        [key_column],
        fail_on_null=True,
    )

    duplicates = (
        df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"Duplicate keys in dimension: {dimension_name}"
        )

    dimension_dfs[dimension_name] = (
        df.select(key_column).distinct()
    )

print("Gold dimension keys validated.")


# ---------------------------------------------------------------------------
# Prepare source data for Gold joins
# ---------------------------------------------------------------------------

# Explicit column selections prevent duplicate field names
# and retain clear metric ownership between FBref sources.

prepared_dfs = {}

all_metric_names = []

for source_name, df in silver_dfs.items():

    mapping = METRIC_MAPPING[source_name]

    all_metric_names.extend(mapping.values())

    prepared_dfs[source_name] = (
        df.select(
            *SILVER_KEYS,
            *[
                F.col(source_column).alias(gold_column)
                for source_column, gold_column in mapping.items()
            ],
        )
    )

# Ensure the target does not contain conflicting metric names.

if len(all_metric_names) != len(set(all_metric_names)):
    raise ValueError(
        "Duplicate Gold metric names found in METRIC_MAPPING."
    )


# ---------------------------------------------------------------------------
# Build consolidated Gold fact
# ---------------------------------------------------------------------------

# Standings is the authoritative source for the fact grain.
# All other FBref datasets are joined on the same business keys.
#
# Left joins retain all standings rows.
# The earlier key validation guarantees complete coverage.

gold_df = prepared_dfs["standings"]

for source_name in [
    "standard",
    "shooting",
    "goalkeeping",
    "playing_time",
    "miscellaneous",
]:

    gold_df = gold_df.join(
        prepared_dfs[source_name],
        on=SILVER_KEYS,
        how="left",
    )

# Convert existing Silver identifiers into shared Gold keys.

gold_df = (
    gold_df
    .withColumn(
        "competition_key",
        F.col("league_code"),
    )
    .withColumn(
        "season_key",
        F.col("season_start_year").cast("int"),
    )
    .select(
        *BUSINESS_KEY,
        *all_metric_names,
    )
)


# ---------------------------------------------------------------------------
# Validate foreign key relationships
# ---------------------------------------------------------------------------

# No fact record may reference a missing Gold dimension member.

foreign_keys = {
    "team_key": dimension_dfs["team"],
    "competition_key": dimension_dfs["competition"],
    "season_key": dimension_dfs["season"],
}

for key_column, dimension_df in foreign_keys.items():

    missing_keys_df = (
        gold_df
        .select(key_column)
        .distinct()
        .join(
            dimension_df,
            on=key_column,
            how="left_anti",
        )
    )

    missing_count = missing_keys_df.count()

    if missing_count > 0:
        display(missing_keys_df)

        raise ValueError(
            f"Gold fact contains {missing_count} "
            f"unknown {key_column} values."
        )

print("All fact foreign keys reference valid Gold dimensions.")


# ---------------------------------------------------------------------------
# Validate Gold fact grain
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

if gold_count != base_count:
    raise ValueError(
        "Gold fact row count does not match standings source: "
        f"expected={base_count}, actual={gold_count}"
    )

duplicate_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    display(duplicate_keys)

    raise ValueError(
        "Duplicate team-season business keys found."
    )


# ---------------------------------------------------------------------------
# Validate football business rules
# ---------------------------------------------------------------------------

# Wins, draws and losses must equal matches played.
# This does not assume that points equal 3 * wins + draws,
# because official standings may include adjustments.

invalid_match_totals = (
    gold_df
    .filter(
        F.col("matches_played")
        != (
            F.col("matches_won")
            + F.col("matches_drawn")
            + F.col("matches_lost")
        )
    )
)

if invalid_match_totals.limit(1).count() > 0:
    display(
        invalid_match_totals.select(
            *BUSINESS_KEY,
            "matches_played",
            "matches_won",
            "matches_drawn",
            "matches_lost",
        )
    )

    raise ValueError(
        "Invalid match totals in team-season standings."
    )

# Goal difference must equal goals scored minus goals conceded.

invalid_goal_difference = (
    gold_df
    .filter(
        F.col("goal_difference")
        != (
            F.col("goals_scored")
            - F.col("goals_conceded")
        )
    )
)

if invalid_goal_difference.limit(1).count() > 0:
    display(
        invalid_goal_difference.select(
            *BUSINESS_KEY,
            "goals_scored",
            "goals_conceded",
            "goal_difference",
        )
    )

    raise ValueError(
        "Invalid goal difference in team-season standings."
    )

# Selected count metrics cannot be negative.
# Goal difference, ratios and standings position are excluded.

NON_NEGATIVE_COLUMNS = [
    "matches_played",
    "matches_won",
    "matches_drawn",
    "matches_lost",
    "goals_scored",
    "goals_conceded",
    "assists",
    "shots",
    "shots_on_target",
    "saves",
    "clean_sheets",
    "yellow_cards",
    "red_cards",
    "fouls_committed",
    "players_used_count",
]

invalid_count_condition = reduce(
    lambda left, right: left | right,
    [
        F.col(column_name) < 0
        for column_name in NON_NEGATIVE_COLUMNS
    ],
)

invalid_counts = (
    gold_df
    .filter(invalid_count_condition)
    .limit(1)
    .count()
)

if invalid_counts > 0:
    raise ValueError(
        "Negative count metrics detected in fact_team_season."
    )

# Shooting accuracy must be a percentage between 0 and 100.

invalid_shooting_percentage = (
    gold_df
    .filter(
        F.col("shots_on_target_percentage").isNotNull()
        & ~F.col("shots_on_target_percentage").between(0, 100)
    )
    .limit(1)
    .count()
)

if invalid_shooting_percentage > 0:
    raise ValueError(
        "Invalid shooting percentage detected."
    )

# Possession must be a percentage between 0 and 100.

invalid_possession = (
    gold_df
    .filter(
        F.col("possession_percentage").isNotNull()
        & ~F.col("possession_percentage").between(0, 100)
    )
    .limit(1)
    .count()
)

if invalid_possession > 0:
    raise ValueError(
        "Invalid possession percentage detected."
    )

print(
    "Gold fact business validation successful: "
    f"{gold_count} team-season records."
)


# ---------------------------------------------------------------------------
# Compare overlapping FBref source metrics
# ---------------------------------------------------------------------------

# Differences between source datasets are reported rather
# than automatically treated as errors.
#
# FBref tables may use different counting conventions.
# Standings remain authoritative for league results.

standard_comparison_df = (
    silver_dfs["standings"]
    .select(
        *SILVER_KEYS,
        F.col("goals_scored").alias("standings_goals"),
        F.col("matches_played").alias("standings_matches"),
    )
    .join(
        silver_dfs["standard"].select(
            *SILVER_KEYS,
            F.col("goals_scored").alias("standard_goals"),
            F.col("matches_played").alias("standard_matches"),
        ),
        on=SILVER_KEYS,
        how="inner",
    )
)

goal_count_differences = (
    standard_comparison_df
    .filter(
        ~F.col("standings_goals").eqNullSafe(
            F.col("standard_goals")
        )
    )
    .count()
)

match_count_differences = (
    standard_comparison_df
    .filter(
        ~F.col("standings_matches").eqNullSafe(
            F.col("standard_matches")
        )
    )
    .count()
)

print(
    "Cross-source metric comparison:"
)

print(
    f"Goals scored differences: {goal_count_differences}"
)

print(
    f"Matches played differences: {match_count_differences}"
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for the current project phase.
# Gold rows are regenerated deterministically from Silver.
#
# The write replaces the table contents and therefore does
# not append duplicate team-season records on reruns.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_metrics = validate_required_not_null(
    saved_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

saved_count = saved_metrics["row_count"]

if saved_count != base_count:
    raise ValueError(
        "Post-write row count mismatch: "
        f"expected={base_count}, actual={saved_count}"
    )

saved_duplicate_keys = (
    saved_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: "
        "duplicate team-season business keys."
    )

# Compare the exact saved business keys to the Silver base.

expected_keys_df = (
    base_keys_df
    .select(
        "team_key",
        F.col("league_code").alias("competition_key"),
        F.col("season_start_year").alias("season_key"),
    )
)

saved_keys_df = saved_df.select(*BUSINESS_KEY)

missing_saved_keys = (
    expected_keys_df
    .join(
        saved_keys_df,
        on=BUSINESS_KEY,
        how="left_anti",
    )
    .count()
)

unexpected_saved_keys = (
    saved_keys_df
    .join(
        expected_keys_df,
        on=BUSINESS_KEY,
        how="left_anti",
    )
    .count()
)

if missing_saved_keys > 0 or unexpected_saved_keys > 0:
    raise ValueError(
        "Post-write keys do not match Silver standings. "
        f"Missing={missing_saved_keys}, "
        f"unexpected={unexpected_saved_keys}"
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD TEAM-SEASON FACT LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Total team-season records: {saved_count}")
print(f"Duplicate business keys: {saved_duplicate_keys}")
print(f"Missing foreign keys: 0")
print(f"Goals scored source differences: {goal_count_differences}")
print(f"Matches played source differences: {match_count_differences}")
print("=" * 55)

display(
    saved_df
    .groupBy("season_key", "competition_key")
    .agg(
        F.count("*").alias("teams"),
        F.sum("matches_played").alias("team_matches"),
        F.sum("goals_scored").alias("total_goals"),
    )
    .orderBy("season_key", "competition_key")
)

display(
    saved_df
    .orderBy(
        "season_key",
        "competition_key",
        "league_position",
    )
)
